In [41]:
!nvidia-smi

Wed Jul  8 18:40:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             31W /   70W |    4163MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [42]:
!pip install -q transformers
!pip install -q datasets
!pip install -q peft
!pip install -q trl
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q sentencepiece

In [43]:
from datasets import load_dataset

dataset = load_dataset(
    "Anupam007/indian-recipe-dataset",
    split="train"
)

In [44]:
print(dataset)

Dataset({
    features: ['TranslatedRecipeName', 'TranslatedIngredients', 'TotalTimeInMins', 'Cuisine', 'TranslatedInstructions', 'URL', 'Cleaned-Ingredients', 'image-url', 'Ingredient-count'],
    num_rows: 5938
})


In [45]:
print(dataset.features)

{'TranslatedRecipeName': Value('string'), 'TranslatedIngredients': Value('string'), 'TotalTimeInMins': Value('int64'), 'Cuisine': Value('string'), 'TranslatedInstructions': Value('string'), 'URL': Value('string'), 'Cleaned-Ingredients': Value('string'), 'image-url': Value('string'), 'Ingredient-count': Value('int64')}


In [46]:
dataset[0]

{'TranslatedRecipeName': 'Masala Karela Recipe',
 'TranslatedIngredients': '1 tablespoon Red Chilli powder,3 tablespoon Gram flour (besan),2 teaspoons Cumin seeds (Jeera),1 tablespoon Coriander Powder (Dhania),2 teaspoons Turmeric powder (Haldi),Salt - to taste,1 tablespoon Amchur (Dry Mango Powder),6 Karela (Bitter Gourd/ Pavakkai) - deseeded,Sunflower Oil - as required,1 Onion - thinly sliced',
 'TotalTimeInMins': 45,
 'Cuisine': 'Indian',
 'TranslatedInstructions': 'To begin making the Masala Karela Recipe,de-seed the karela and slice.\nDo not remove the skin as the skin has all the nutrients.\nAdd the karela to the pressure cooker with 3 tablespoon of water, salt and turmeric powder and pressure cook for three whistles.\nRelease the pressure immediately and open the lids.\nKeep aside.Heat oil in a heavy bottomed pan or a kadhai.\nAdd cumin seeds and let it sizzle.Once the cumin seeds have sizzled, add onions and saute them till it turns golden brown in color.Add the karela, red chi

teaching the llm to how to take data as it cant take row and columns (user and assistent)

In [47]:
recipe = dataset[0]

for key, value in recipe.items():
    print("=" * 50)
    print(key)#key value pairs like dictionary in python
    print(value)

TranslatedRecipeName
Masala Karela Recipe
TranslatedIngredients
1 tablespoon Red Chilli powder,3 tablespoon Gram flour (besan),2 teaspoons Cumin seeds (Jeera),1 tablespoon Coriander Powder (Dhania),2 teaspoons Turmeric powder (Haldi),Salt - to taste,1 tablespoon Amchur (Dry Mango Powder),6 Karela (Bitter Gourd/ Pavakkai) - deseeded,Sunflower Oil - as required,1 Onion - thinly sliced
TotalTimeInMins
45
Cuisine
Indian
TranslatedInstructions
To begin making the Masala Karela Recipe,de-seed the karela and slice.
Do not remove the skin as the skin has all the nutrients.
Add the karela to the pressure cooker with 3 tablespoon of water, salt and turmeric powder and pressure cook for three whistles.
Release the pressure immediately and open the lids.
Keep aside.Heat oil in a heavy bottomed pan or a kadhai.
Add cumin seeds and let it sizzle.Once the cumin seeds have sizzled, add onions and saute them till it turns golden brown in color.Add the karela, red chilli powder, amchur powder, coriander

take the required ingredients

In [48]:
def create_prompt(example):

    prompt = f"""
User:
How do I make {example['TranslatedRecipeName']}?

Assistant:
Ingredients:
{example['TranslatedIngredients']}

Instructions:
{example['TranslatedInstructions']}
"""

    return {"text": prompt}

If the dataset has

6000 recipes

this function will be called

6000 times.

so we are applying to entire dataset.

In [49]:
dataset = dataset.map(create_prompt)

In [50]:
print(dataset[15]["text"])


User:
How do I make South Indian Onion Chutney Recipe - South Indian Onion Chutney (Recipe)?

Assistant:
Ingredients:
 1 teaspoon cumin seeds, 2 teaspoons oil, 2 tablespoons black urad dal (split), salt - 1 teaspoon, 3 dry red chillies, 1/2 teaspoon oil, 1 tablespoon tamarind paste, as per taste Rye, 1 sprig curry leaves, 1/2 teaspoon jaggery,2 onions

Instructions:
To make South Indian Onion Chutney, first of all chop the onion and keep it aside.
Now heat 1 teaspoon of oil in the pan.
Add cumin seeds, dry red chillies and let it cook for 10 seconds.
Now add urad dal in it and let it cook till it becomes golden.
Turn off the gas and drain in a bowl.
Add another spoon of oil to the same pan.
Add onions and let it cook for 4 to 5 minutes.
Turn off the gas and let it cool down.
Now put urad dal, cumin and dry red chillies in the mixer grinder and grind them.
Add cooked onions, tamarind and jaggery.
Add some water and grind it.
Now for the tempering, heat the oil in a small pan.
Add musta

loading the model

In [51]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer

In [52]:
text = "How do I make dosa?"

tokens = tokenizer(text)

print(tokens)

{'input_ids': [4340, 653, 358, 1281, 8750, 64, 30], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


In [53]:
decoded = tokenizer.decode(tokens["input_ids"])

print(decoded)

How do I make dosa?


pre training a step just before fine tune

In [54]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

we can compair with the results

In [64]:
messages = [
    {
        "role": "user",
        "content": "How do I make Paneer Butter Masala?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
How do I make Paneer Butter Masala?
assistant
Paneer Butter Masala is a popular Indian dish that's creamy, flavorful, and delicious. Here’s a simple recipe to make it:

### Ingredients:
- **Paneer (Indian cheese)**: 200-30ml (about 150g)
- **Butter**: 2 tbsp (30g)
- **Tomato paste or puree**: 1/4 cup (60ml)
- **Ginger-garlic paste**: 1 tsp
- **Turmeric powder**: 1/2 tsp
- **Cumin seeds**: 1/2 tsp
- **Curry leaves**: A few
- **Red chili flakes**: 1/2 tsp (optional, for extra heat)
- **Coconut milk**: 200ml (for the sauce)
- **Water**: 1/2 cup (120ml)
- **Salt**: To taste
- **Chopped fresh coriander**: For


fine turning starts

In [56]:
def create_messages(example):
    return {
        "messages": [
            {
                "role": "user",
                "content": f"How do I make {example['TranslatedRecipeName']}?"
            },
            {
                "role": "assistant",
                "content":
                f"""Ingredients:
{example['TranslatedIngredients']}

Instructions:
{example['TranslatedInstructions']}"""
            }
        ]
    }

In [57]:
dataset = dataset.map(create_messages)

model always trains like this:
<|im_start|>user
Question
<|im_end|>

<|im_start|>assistant
Answer
<|im_end|>

In [58]:
dataset[22]["messages"]

[{'role': 'user',
  'content': 'How do I make Homemade Healthy Subway Sandwich Recipe With Beet & Sprout?'},
 {'role': 'assistant',
  'content': 'Ingredients:\n2 Tomatoes - thinly sliced,1 Beetroot - grated,1/2 cup Hung Curd (Greek Yogurt),2 Submarine Bread (Subway Bread) - (flat breads or foot longs),2 cloves Garlic - grated,Tabasco Original - Hot Sauce - to taste,Salt and Pepper - to taste,2 Stalks Spring Onion (Bulb & Greens) - finely chopped,1/2 cup Green Moong Sprouts,2 tablespoon Coriander (Dhania) Leaves - finely chopped\n\nInstructions:\nTo begin making Subway Sandwich Recipe With Roasted Beetroot, we will first cook the beets.Heat a teaspoon of oil in a wok; add the grated beets and garlic, sprinkle some salt and stir fry on low to medium heat until the beets are softened and cooked.Once you notice the beetroot is cooked through, turn off the heat and allow it to cool completely.Once the beetroot has cooled completely, add it to a large mixing bowl.\nTo this add in the greek y

In [59]:
text = tokenizer.apply_chat_template(
    dataset[22]["messages"],
    tokenize=False
)

print(text)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
How do I make Homemade Healthy Subway Sandwich Recipe With Beet & Sprout?<|im_end|>
<|im_start|>assistant
Ingredients:
2 Tomatoes - thinly sliced,1 Beetroot - grated,1/2 cup Hung Curd (Greek Yogurt),2 Submarine Bread (Subway Bread) - (flat breads or foot longs),2 cloves Garlic - grated,Tabasco Original - Hot Sauce - to taste,Salt and Pepper - to taste,2 Stalks Spring Onion (Bulb & Greens) - finely chopped,1/2 cup Green Moong Sprouts,2 tablespoon Coriander (Dhania) Leaves - finely chopped

Instructions:
To begin making Subway Sandwich Recipe With Roasted Beetroot, we will first cook the beets.Heat a teaspoon of oil in a wok; add the grated beets and garlic, sprinkle some salt and stir fry on low to medium heat until the beets are softened and cooked.Once you notice the beetroot is cooked through, turn off the heat and allow it to cool completely.Once the beetroot has cooled

LoRA(low rank adaptation)
in 3B paramater we would only train 5 to 10M parameters
that is less that 1% of the model

In [60]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [61]:
from peft import get_peft_model

model = get_peft_model(model, peft_config)

In [62]:
model.print_trainable_parameters()

trainable params: 7,372,800 || all params: 3,093,311,488 || trainable%: 0.2383


dataset after fine tuning


In [63]:
def formatting_func(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False
        )
    }

dataset = dataset.map(formatting_func)

In [65]:
print(dataset[14]["text"])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
How do I make Saunf Aloo (Fennel Potato Curry) Recipe?<|im_end|>
<|im_start|>assistant
Ingredients:
1/4 cup Fresh cream - or 1/2 cup milk,5 Potatoes (Aloo) - halved with skin,2 teaspoon Fennel seeds (Saunf) - crushed,4 sprig Coriander (Dhania) Leaves - finely chopped,1 teaspoon Turmeric powder (Haldi),2 teaspoon Red Chilli powder

Instructions:
To begin with Saunf Aloo, heat oil in a pressure cooker.
Add turmeric powder, salt, red chilli powder and crushed fennel seeds till the fennel seeds turns golden in colour.Now add the potatoes and mix it properly with the masala.
Sauce it for 2-3 minutes and then add enough water to cover the potatoes in the cooker.Switch the heat after two whistles and once the pressure is released, open the lid of the pressure cooker.The next step is to add the cream or milk and mash a few potatoes to get a thicker consistency.
Let it cook for 5 m

In [66]:
columns_to_keep = ["text"]

dataset = dataset.remove_columns(
    [col for col in dataset.column_names if col not in columns_to_keep]
)

In [67]:
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 5938
})
